# 📊 Gold Spot
Notebook 02: Modeling

## Preliminaries


In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

from src.config import clean_data_file
from src.modeling_functions import *

In [2]:
# Open data file
df = pd.read_parquet(clean_data_file)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 875 entries, 0 to 874
Data columns (total 11 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   date                          875 non-null    datetime64[ns]
 1   real_yield_10y_lag1           875 non-null    float64       
 2   breakeven_inflation_10y_lag1  875 non-null    float64       
 3   dxy_broad_goods_lag1          875 non-null    float64       
 4   treasury_10y_nominal_lag1     875 non-null    float64       
 5   gold_return                   875 non-null    float64       
 6   oil_return                    875 non-null    float64       
 7   spy_return                    875 non-null    float64       
 8   target_gold_t1                874 non-null    float64       
 9   vix_index_lag1_log            875 non-null    float64       
 10  month_of_year                 875 non-null    int32         
dtypes: datetime64[ns](1), float64(9)

In [ ]:
# Ensure order by date
df.sort_values(by='date', inplace=True)

# Split training and live data
target = 'target_gold_t1'
features_to_remove = []
X_train, X_live, y_train, _ = split_last(df, label=target, cols_excluded=features_to_remove)

# Column transformer
cyclical_features = ["month_of_year"]
numeric_features = [col for col in X_train.select_dtypes(include=np.number) if col not in cyclical_features and col != 'date']

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cyclical", FunctionTransformer(encode_cyclical_month), cyclical_features),
    ]
)

# Base model pipeline
model_name = 'Ridge'
pipe = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", Ridge(alpha=10.0)),  # alpha modera la sensibilidad al ruido
    ]
)

# Training
pipe.fit(X_train, y_train)

# Predict
y_pred = pipe.predict(X_live)
print(f'Training completed.\nPrediction: {y_pred}')

Training completed.


In [6]:
# Cross validation test
tscv = TimeSeriesSplit(n_splits=5)

cross_val_scores = cross_val_score(
    estimator=pipe, 
    X=X_train, 
    y=y_train, 
    cv=tscv,         
    scoring='d2_absolute_error_score',
    n_jobs=-1        
)

print(f"Average Score Time Series CV: {cross_val_scores.mean():.4f} ± {cross_val_scores.std():.4f}")

Average Score Time Series CV: -0.0438 ± 0.0310
